In [1]:
# === Imports ===

from dataclasses import dataclass, field
from math import ceil as pyceil
from typing import Dict, Literal, Optional, Tuple, Type

import numpy as np
from matplotlib import pyplot as plt
from matplotlib import style as mplstyle
from numba import jit
from scipy.fft import next_fast_len
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import LinearOperator, lsqr

mplstyle.use("./docs/pyscopee.mplstyle")

%matplotlib widget

In [2]:
# the regularly sampled signal is loaded
data_regular = np.loadtxt("signal_regular.txt", delimiter=",", skiprows=1)

t_values_regular = data_regular[:, 0]
y_values_regular = data_regular[:, 1]

# the irregularly sampled signal is loaded
data_irregular = np.loadtxt("signal_irregular_with_noise.txt", delimiter=",", skiprows=1)

t_values_irregular = data_irregular[:, 0]
y_values_irregular = data_irregular[:, 2]
noise_stddevs_irregular = data_irregular[:, 3]

In [ ]:
# === Interpolation Constants ===

BASE_VANDER_PINVS: Dict[Literal[1, 3, 5], np.ndarray] = {
    poly_degree: np.linalg.pinv(
        np.vander(
            np.linspace(start=-1, stop=1, num=poly_degree + 1),
            N=poly_degree + 1,
            increasing=True,
        )
    )
    for poly_degree in (1, 3, 5)
}


# === Interpolation Functions ===

# @jit(nopython=True)
def fast_vandermonde_matrix(x: np.ndarray, poly_degree: int) -> np.ndarray:
    """
    Equivalent to ``numpy.vander(x, N=poly_degree + 1, increasing=True)``, but faster.

    """
    # the Vandermonde matrix is initialised in its transposed form because then the
    # rows are contiguous in memory, which is beneficial for performance
    vandermonde_matrix = np.empty((poly_degree + 1, x.size), dtype=x.dtype)
    vandermonde_matrix[0, ::] = 1.0

    # all rows involving x are calculated iteratively with direct memory overwrites
    for power in range(1, poly_degree + 1):
        np.multiply(vandermonde_matrix[power - 1, ::], x, vandermonde_matrix[power, ::],)

    return vandermonde_matrix.transpose()


# @jit(nopython=True)
def sorted_searchsorted_left(a: np.ndarray, v: np.ndarray) -> np.ndarray:
    """
    Equivalent to ``numpy.searchsorted(a, v, side="left")``, but faster.

    It exploits the fact that both ``a`` and ``v`` are sorted in ascending order
    and that ``v[0] == a[0]`` and ``v[-1] == a[-1]``.

    """

    result = np.empty_like(v, dtype=np.int64)
    v_index = 0
    reached_end = False
    for a_index, a_value in enumerate(a):
        while v[v_index] <= a_value:
            result[v_index] = a_index
            v_index += 1

            if v_index >= v.size:
                reached_end = True
                break

        if reached_end:
            break

    return result


# @jit(nopython=True)
def _make_interpolation_matrix_csr_specs(
    t_grid: np.ndarray,
    t_eval: np.ndarray,
    poly_degree: Literal[1, 3, 5],
    base_vander_pinv: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Creates the specifications for a CSR matrix ``A`` that interpolates a signal given
    at ``t_grid`` at the time points ``t_eval`` using a piecewise polynomial of degree
    ``poly_degree``.

    Parameters
    ----------
    base_vander_pinv : :class:`numpy.ndarray` of shape ``(poly_degree + 1, poly_degree + 1)``
        The Moore Penrose pseudoinverse of the Vandermonde matrix of the polynomial
        basis with ``poly_degree + 1`` grid points evenly spaced  in the interval
        ``[-1, 1]``.

    For the other parameters, please refer to the function :func:`make_interpolation_matrix`.

    Returns
    -------
    data : :class:`numpy.ndarray` of shape ``(m * (poly_degree + 1),)``
        The non-zero entries of the CSR matrix ``A``.
    indices : :class:`numpy.ndarray` of shape ``(m * (poly_degree + 1),)``
        The column indices of the non-zero entries of the CSR matrix ``A``.
    indptr : :class:`numpy.ndarray` of shape ``(m + 1,)``
        The indices in the ``data`` and ``indices`` arrays where the data for each row
        starts and ends.

    Notes
    -----
    Given that the dense version of ``A`` fits into memory, the dense matrix can be
    created with

    ```python
    A_dense = np.zeros(shape=A.shape, dtype=np.float64)
    for row_index in range(A.shape[0]):
        ptr_from = indptr[row_index]
        ptr_to = indptr[row_index + 1]
        A_dense[row_index, indices[ptr_from:ptr_to]] = data[ptr_from:ptr_to]
    ```

    """  # noqa: E501

    # the first indices grid point of the respective polynomial basis is found for
    # each evaluation time point;
    # ``sorted_searchsorted_left`` finds the indices ``a_index`` such that
    # ``t_grid[a_index - 1] < t_eval <= t_grid[a_index]``;
    # since only odd order polynomials are considered, this interval corresponds to
    # the two central grid points of the respective polynomial and therefore
    # ``a_index - (poly_degree + 1) // 2`` is the first grid point of the polynomial
    # NOTE: for the first and last evaluation time points, the polynomial basis indices
    #       will be out of bounds and are therefore clipped to the valid range
    poly_degree_plus_one = poly_degree + 1
    base_indices = np.clip(
        sorted_searchsorted_left(a=t_grid, v=t_eval) - (poly_degree + 1) // 2,
        0,
        t_grid.size - poly_degree_plus_one,
    )

    # next, the evaluation time points are mapped to the interval ``[-1, 1]`` of their
    # respective polynomial and the interpolation weights for each of the respective
    # ``poly_degree + 1`` grid points are calculated by a simple matrix multiplication
    # with the Moore Penrose pseudoinverse of the base Vandermonde matrix
    half_poly_base_width = (
        0.5 * poly_degree * ((t_grid[-1] - t_grid[0]) / (t_grid.size - 1))
    )
    data = (
        fast_vandermonde_matrix(
            (t_eval - (t_grid[base_indices] + half_poly_base_width))
            / half_poly_base_width,
            poly_degree=poly_degree,
        )
        @ base_vander_pinv
    ).ravel()

    # the column indices are simply the indices of the grid points of the respective
    # polynomial basis
    indices = (
        base_indices[::, np.newaxis]
        + np.arange(
            0,
            poly_degree_plus_one,
            1,
            dtype=np.int64,
        )
    ).ravel()

    # the index pointers are the indices where the data for each row starts and ends,
    # so here its just the multiples of the number of grid points of the polynomials,
    # i.e., ``poly_degree + 1``
    indptr = np.arange(
        0,
        (t_eval.size + 1) * poly_degree_plus_one,
        poly_degree_plus_one,
        dtype=np.int64,
    )

    return data, indices, indptr


def make_interpolation_matrix(
    t_grid: np.ndarray,
    t_eval: np.ndarray,
    poly_degree: Literal[1, 3, 5],
) -> csr_matrix:
    """
    Creates a CSR matrix ``A`` that interpolates a signal given at ``t_grid`` at the
    time points ``t_eval`` using a piecewise polynomial of degree ``poly_degree``.
    If the signal is given as a vector ``y``, the interpolated signal can be obtained
    by ``A @ y``.

    Parameters
    ----------
    t_grid : :class:`numpy.ndarray` of shape ``(n,)``
        The sorted time points at which the signal is given.
        It is assumed that ``t_grid[0] == t_eval[0]`` and ``t_grid[-1] == t_eval[-1]``.
    t_eval : :class:`numpy.ndarray` of shape ``(m,)``
        The sorted time points at which the signal should be interpolated.
    poly_degree : {``1``, ``3``, ``5``}
        The degree of the piecewise polynomial used for interpolation.

    Returns
    -------
    A : :class:`scipy.sparse.csr_matrix` of shape ``(m, n)``
        The interpolation matrix.

    """

    try:
        base_vander_pinv = BASE_VANDER_PINVS[poly_degree]
    except KeyError:
        raise ValueError(
            f"Polynomial degree must be one of 1, 3, or 5, but got {poly_degree}."
        )

    return csr_matrix(
        _make_interpolation_matrix_csr_specs(
            t_grid=t_grid,
            t_eval=t_eval,
            poly_degree=poly_degree,
            base_vander_pinv=base_vander_pinv,
        ),
        shape=(t_eval.size, t_grid.size),
    )


def get_t_grid_for_decimation(
    t_values: np.ndarray,
    decimation_factor: float,
    use_fft_next_fast_len: bool = True,
) -> np.ndarray:
    """
    Returns a grid of equally spaced time points for decimation.

    Parameters
    ----------
    t_values : :class:`numpy.ndarray` of shape ``(n,)``
        The time points for which the grid is created.
    decimation_factor : :class:`float`
        The factor by which the number of time points is reduced.
    use_fft_next_fast_len : :class:`bool`, default=``True``
        If ``False``, the number of grid points is given as the next closest integer
        greater than or equal to ``n / decimation_factor``.
        If ``True``, the number of grid points will be the next fast length for the
        Fourier Transform (``numpy.fft.fft``) greater than or equal to
        ``n / decimation_factor``.

    Returns
    -------
    t_grid : :class:`numpy.ndarray` of shape ``(m,)``
        The grid of equally spaced time points for decimation.

    """

    num_grid_points = pyceil(t_values.size / decimation_factor)
    if use_fft_next_fast_len:
        num_grid_points = next_fast_len(num_grid_points, real=False)

    return np.linspace(
        start=t_values[0],
        stop=t_values[-1],
        num=num_grid_points,  # type: ignore
    )


a = np.linspace(-1, 1, 5_000)
v = np.linspace(-1, 1, 50_000)

res1 = np.searchsorted(a, v, side="left")
res2 = sorted_searchsorted_left(a, v)

assert np.array_equal(res1, res2)
# %timeit np.searchsorted(a, v, side="left")
# %timeit sorted_searchsorted_left(a, v)

test_poly_degree = 3
res1 = np.vander(a, N=test_poly_degree + 1, increasing=True)
res2 = fast_vandermonde_matrix(a, poly_degree=test_poly_degree)

assert np.allclose(res1, res2)

# %timeit np.vander(a, N=test_poly_degree + 1, increasing=True)
# %timeit fast_vandermonde_matrix(a, poly_degree=test_poly_degree)

t_eval = np.linspace(-1, 1, 50_000)
t_grid = np.linspace(-1, 1, 5_000)

A = make_interpolation_matrix(t_grid=t_grid, t_eval=t_eval, poly_degree=3)
A_t = A.T.tocsr()
print(type(A.T))
print(np.round(A.toarray(), 3))

# %timeit make_interpolation_matrix(t_grid=t_grid, t_eval=t_eval, poly_degree=3)
# %timeit at = A.T.tocsr()
# %timeit at = A @ t_grid
# %timeit at = A_t @ t_eval

In [ ]:
# === Decimation Models ===


@dataclass
class Bandlimits:
    """
    A dataclass that contains the lower and upper bandlimit frequencies of the Fourier
    Transform.

    It also holds a flag whether the lower bandlimit is zero and an effective lower
    bandlimit that is used for comparisons where a lower bandlimit of zero has to be
    excluded from the consideration.

    """

    low: float
    high: float
    nonzero_low: float

    low_is_zero: bool = field(init=False)

    def __post_init__(self) -> None:
        """
        Validates the bandlimit frequencies.

        """

        if self.low >= self.high:
            raise ValueError(
                f"The lower bandlimit {self.low:.5e} exceeds the upper bandlimit "
                f"{self.high:.5e}."
            )

        if self.low < 0.0:
            raise ValueError(f"The lower bandlimit {self.low:.5e} is less than zero.")

        self.low_is_zero = self.low == 0.0

        # the effective lower bandlimit is not set yet because this requires the
        # frequencies of the Fourier Transform


@dataclass
class FourierSpecs:
    """
    A dataclass that contains

    - the positive frequencies of the Fourier Transform (negative frequencies are
        given by the Hermitian symmetry of the Fourier Transform for real-valued
        signals)
    - the lower and upper bandlimit
    - whether the lower bandlimit is zero
    - the effective lower bandlimit
    - the size of the coefficient vector x for the Fourier Transform

    """

    freqs: np.ndarray
    bandlimits: Bandlimits
    freq_indices_in_nonzero_bandlimit_from: int = field(init=False)
    freq_indices_in_nonzero_bandlimit_to: int = field(init=False)
    freq_indices_zero_coeffs_below_low_to: int = field(init=False)
    x_vect_size: int = field(init=False)
    x_real_imag_split_index: int = field(init=False)

    def __post_init__(self) -> None:
        """
        Evaluates where the real and imaginary coefficient of the Fourier coefficients
        are located within the bandlimit at the positive frequencies of the Fourier
        Transform.

        Besides, it also calculates the size of the coefficient vector x for the Fourier
        Transform and how the mapping between x and the real and imaginary parts of the
        Fourier coefficients is done.

        """

        # the indices of the coefficients that are located at all NONZERO frequencies
        # within the bandlimit are found
        self.freq_indices_in_nonzero_bandlimit_from = int(
            np.searchsorted(a=self.freqs, v=self.bandlimits.nonzero_low, side="left")
        )
        self.freq_indices_in_nonzero_bandlimit_to = int(
            np.searchsorted(a=self.freqs, v=self.bandlimits.high, side="right")
        )

        # for the indices of the coefficients below the lower bandlimit, there has to be
        # a distinction between the case when the lower bandlimit is zero and when it
        # is not
        self.freq_indices_zero_coeffs_below_low_to = (
            self.freq_indices_in_nonzero_bandlimit_from
            if not self.bandlimits.low_is_zero
            else 0
        )

        # for each nonzero positive frequency, there are 2 coefficients (real and
        # imaginary part);
        # for a potentially zero lower bandlimit, there has to be an additional
        # real coefficient for the zero frequency;
        # this is also accounted for in the split between the real and imaginary
        # coefficients in x
        if not self.bandlimits.low_is_zero:
            x_vect_size_offset = 0
        else:
            x_vect_size_offset = 1

        num_freqs_in_nonzero_bandlimit = (
            self.freq_indices_in_nonzero_bandlimit_to
            - self.freq_indices_in_nonzero_bandlimit_from
        )
        self.x_real_imag_split_index = (
            x_vect_size_offset + num_freqs_in_nonzero_bandlimit
        )
        self.x_vect_size = x_vect_size_offset + 2 * num_freqs_in_nonzero_bandlimit

        return

    def __len__(self) -> int:
        """
        Returns the number of samples in the time domain which is equal to the size of
        the full frequencies of the Fourier Transform.

        """

        return self.freqs.size


# === Decimation Functions ===


def get_fourier_specs(
    t_grid: np.ndarray,
    bandlimit_low: float,
    bandlimit_high: float,
) -> FourierSpecs:
    """
    Computes the Fourier Transform specifications for the given time vector and
    bandlimit frequencies.

    Parameters
    ----------
    t_grid : :class:`numpy.ndarray`
        The time vector of equidistant time points at which the signal is sampled.
    bandlimit_low, bandlimit_high : :class:`float`
        The lower and upper bandlimit frequencies of the Fourier Transform.

    Returns
    -------
    fourier_specs : :class:`FourierSpecs`
        The Fourier Transform specifications for the given time vector and bandlimit
        frequencies. Please refer to the class :class:`FourierSpecs` for more details.

    Raises
    ------
    ValueError
        If there are no frequencies located between the lower and upper bandlimit.
    ValueError
        If the upper bandlimit does not lead to the exclusion of the Nyquist frequency.

    """

    # the positive frequencies of the Fourier Transform are computed
    delta_t = (t_grid[-1] - t_grid[0]) / (t_grid.size - 1)
    freqs = np.fft.rfftfreq(n=t_grid.size, d=delta_t)  # type: ignore

    # the bandlimit frequencies are validated for themselves
    # NOTE: the effective lower bandlimit is clipped to 0.5 times the first positive
    #       frequency to exclude a potentially zero bandlimit frequency from all related
    #       comparisons
    bandlimits = Bandlimits(
        low=bandlimit_low,
        high=bandlimit_high,
        nonzero_low=max(0.5 * freqs[1], bandlimit_low),
    )

    # next, the bandlimits need to be validated against the frequencies of the
    # Fourier Transform
    # then, it is checked if there are any frequencies within the bandlimit
    if not np.any((bandlimits.low <= freqs) & (freqs <= bandlimits.high)):
        raise ValueError(
            f"There are no frequencies within the bandlimit "
            f"[{bandlimits.low:.5e}, {bandlimits.high:.5e}]."
        )

    # afterwards, the upper bandlimit is checked against the Nyquist frequency which
    # should be excluded
    if bandlimits.high >= freqs[-1]:
        raise ValueError(
            f"The upper bandlimit {bandlimits.high:.5e} does not lead to the exclusion "
            f"of the Nyquist frequency {freqs[-1]:.5e}."
        )

    # finally, the Fourier Transform specifications are returned
    return FourierSpecs(
        freqs=freqs,
        bandlimits=bandlimits,
    )


# @jit(nopython=True)
def convert_x_to_real_ft_coefficients(
    x: np.ndarray,
    num_freqs: int,
    x_real_imag_split_index: int,
    bandlimit_low_is_zero: bool,
    freq_indices_in_nonzero_bandlimit_from: int,
    freq_indices_in_nonzero_bandlimit_to: int,
    freq_indices_zero_coeffs_below_low_to: int,
) -> np.ndarray:
    """
    Converts the compressed coefficients ``x`` for the Fourier Transform to the real and
    imaginary coefficients of the Fourier Transform.

    Parameters
    ----------
    x : :class:`numpy.ndarray` of shape (p,)
        The coefficients for the Fourier Transform stored in compressed form as real
        values.
    num_freqs : :class:`int`
        The number of frequencies of the Fourier Transform.
    x_real_imag_split_index : :class:`int`
        The index at which the real and imaginary coefficients are split in the
        coefficient vector ``x``.
        So, the real coefficients can be found at ``x[0:x_real_imag_split_index]``
        while the imaginary coefficients are located at ``x[x_real_imag_split_index:]``.
    bandlimit_low_is_zero : :class:`bool`
        A flag that indicates whether the lower bandlimit is zero (``True``) or not
        (``False``).
    freq_indices_in_nonzero_bandlimit_from, freq_indices_in_nonzero_bandlimit_to : :class:`int`
        The index from the first to the last positive nonzero frequency within the
        bandlimit. For obtaining the nonzero frequencies within the bandlimit, these
        indices can be used as ``freqs[freq_indices_in_nonzero_bandlimit_from:freq_indices_in_nonzero_bandlimit_to]``.
    freq_indices_zero_coeffs_below_low_to : :class:`int`
        The index up to which the frequencies lie below the lower bandlimit (if any).
        For obtaining the frequencies below the lower bandlimit, these indices can be
        used as ``freqs[0:freq_indices_zero_coeffs_below_low_to]``.

    Returns
    -------
    x_real_ft : :class:`numpy.ndarray` of shape (floor(m/2) + 1,)
        The real coefficients of the Fourier Transform as if they were obtained
        by a real Fourier Transform like ``np.fft.rfft``.

    """  # noqa: E501

    # the result is initialised as an empty Array because filling it fully with zeros
    # would be inefficient because the bandlimit would be overwritten afterwards
    x_real_ft = np.empty(shape=(num_freqs,), dtype=np.complex128)

    # if the lower bandlimit is not zero, leading zeros are filled at the indices below
    # the lower bandlimit
    if not bandlimit_low_is_zero:
        x_offset_index = 0
        x_real_ft[0:freq_indices_zero_coeffs_below_low_to] = 0.0

    # otherwise, if the lower bandlimit is zero, the first coefficient is the real
    # coefficient for the zero frequency
    else:
        x_offset_index = 1
        x_real_ft[0] = x[0]

    # the real and imaginary coefficients are split and stored in the result
    x_real_ft[
        freq_indices_in_nonzero_bandlimit_from:freq_indices_in_nonzero_bandlimit_to
    ] = (
        x[x_offset_index:x_real_imag_split_index]
        + 1.0j * x[x_real_imag_split_index : x.size]
    )

    # finally, the trailing zeros are filled at the indices above the upper bandlimit
    x_real_ft[freq_indices_in_nonzero_bandlimit_to : x_real_ft.size] = 0.0

    return x_real_ft


# @jit(nopython=True)
def compress_ft_coefficients_to_xlike(
    x_real_ft: np.ndarray,
    x_vect_size: int,
    x_real_imag_split_index: int,
    bandlimit_low_is_zero: bool,
    freq_indices_in_nonzero_bandlimit_from: int,
    freq_indices_in_nonzero_bandlimit_to: int,
) -> np.ndarray:
    """
    Compresses the real and imaginary coefficients of the Fourier Transform to a
    compressed form as real values similar (but not fully identical) to the real
    x-coefficients representing the Fourier Transform.

    For the parameters, please refer to the function :func:`convert_x_to_real_ift_coefficients`.

    """  # noqa: E501

    # the result is initialised as an empty Array because it will not have a single
    # zero value
    x = np.empty(shape=(x_vect_size,), dtype=np.float64)

    # if the lower bandlimit is zero, the first coefficient is the real coefficient for
    # the zero frequency
    x_offset_index = 0
    if bandlimit_low_is_zero:
        x[0] = x_real_ft.real[0]
        x_offset_index = 1

    # then, the twofold of the real coefficients are stored in the first half of x
    x[x_offset_index:x_real_imag_split_index] = (
        2.0
        * x_real_ft[
            freq_indices_in_nonzero_bandlimit_from:freq_indices_in_nonzero_bandlimit_to
        ].real
    )

    # finally, the negative twofold of the imaginary coefficients are stored in the
    # second half of x
    x[x_real_imag_split_index : x.size] = (
        -2.0
        * x_real_ft[
            freq_indices_in_nonzero_bandlimit_from:freq_indices_in_nonzero_bandlimit_to
        ].imag
    )

    return x


# === Decimation Classes ===


class LinearOperatorForBandlimitedDecimation(LinearOperator):
    """
    A class that represents a linear operator for bandlimited decimation.

    Parameters
    ----------
    t_values : :class:`numpy.ndarray` of shape (n,)
        The sorted time points at which the signal is sampled.
        They don't have to be equidistantly spaced, but they have to be sorted.
    t_grid_for_ft : :class:`numpy.ndarray` of shape (m,)
        The grid of equally spaced time points for the Fourier Transform.
        It is assumed to be equidistantly spaced, sorted, and that
        ``t_grid_for_ft[0] == t_values[0]`` and ``t_grid_for_ft[-1] == t_values[-1]``.
    bandlimit_high : :class:`float`
        The upper bandlimit frequency of the Fourier Transform.
    bandlimit_low : :class:`float`, default=0.0
        The lower bandlimit frequency of the Fourier Transform.
    y_uncertainties : :class:`numpy.ndarray` of shape (m,) or ``None``, default=``None``
        The uncertainties of the signal values at the time points which can be used for
        a Weighted Least Squares problem.
        If ``None``, the signal values are assumed to have equal uncertainties and an
        Ordinary Least Squares problem is solved.

    """

    def __init__(
        self,
        t_values: np.ndarray,
        t_grid_for_ft: np.ndarray,
        bandlimit_high: float,
        bandlimit_low: float = 0.0,
        y_uncertainties: Optional[np.ndarray] = None,
        interpolate_polynomial_degree: Literal[1, 3, 5] = 3,
    ) -> None:

        # the Fourier Transform specifications are obtained
        self.original_size: int = t_values.size
        self.ft_size: int = t_grid_for_ft.size
        self.fourier_specs: FourierSpecs = get_fourier_specs(
            t_grid=t_grid_for_ft,
            bandlimit_low=bandlimit_low,
            bandlimit_high=bandlimit_high,
        )

        # the interpolation matrix is created if the time points for the signal values
        # are not the same as the time points for the Fourier Transform
        if not np.array_equiv(t_values, t_grid_for_ft):
            self.interpolation_matrix: csr_matrix = make_interpolation_matrix(
                t_grid=t_grid_for_ft,
                t_eval=t_values,
                poly_degree=interpolate_polynomial_degree,
            )
            self.interpolation_matrix_transposed: csr_matrix = (
                self.interpolation_matrix.T.tocsr()
            )
            self.interpolation_required: bool = True

        else:
            self.interpolation_matrix = csr_matrix((0, 0), dtype=np.float64)
            self.interpolation_matrix_transposed = csr_matrix((0, 0), dtype=np.float64)
            self.interpolation_required = False

        # the weights are computed from the uncertainties (if provided) and the
        # respective matrix-vector products are defined to avoid an if-statement in the
        # ``_matvec`` and ``_rmatvec`` methods
        if y_uncertainties is not None:
            self.sqrt_weights: np.ndarray = np.reciprocal(y_uncertainties)
            self.is_weighted_problem: bool = True
        else:
            self.sqrt_weights: np.ndarray = np.array([], dtype=np.float64)
            self.is_weighted_problem: bool = False

        self.shape: Tuple[int, int] = (
            self.original_size,
            self.fourier_specs.x_vect_size,
        )
        self.dtype: Type = np.float64

    def _apply_sqrt_weights(self, x: np.ndarray) -> np.ndarray:
        """
        Applies the square root of the diagonal matrix of the weights to the vector
        ``x``.
        If no weights are provided, the vector ``x`` is returned as is.

        """

        if self.is_weighted_problem:
            return self.sqrt_weights * x

        return x

    def _interpolate(self, x: np.ndarray) -> np.ndarray:
        """
        Interpolates the vector ``x`` to the time points where the signal is actually
        sampled.
        If no interpolation is required, the vector ``x`` is returned as is.

        """

        if self.interpolation_required:
            return self.interpolation_matrix @ x

        return x

    def _interpolate_transposed(self, x: np.ndarray) -> np.ndarray:
        """
        Computes the transpose of the interpolation matrix applied to the vector ``x``.
        If no interpolation is required, the vector ``x`` is returned as is.

        """

        if self.interpolation_required:
            return self.interpolation_matrix_transposed @ x

        return x

    def _matvec(self, x: np.ndarray) -> np.ndarray:
        """
        This is equivalent to the product ``sqrt(W) @ A @ inv(F) @ P @ x`` where

        - ``sqrt(W)`` is the square root of the diagonal matrix of the weights
        - ``A`` is the interpolation matrix that interpolates the signal values from the
            grid of time points to the time points where the signal is actually sampled
        - ``inv(F)`` is the Inverse Fourier Transform
        - ``P`` is the matrix that maps the compressed Fourier coefficients stored in
            ``x`` to the real and imaginary coefficients of the Fourier Transform at
            both the positive and negative frequencies

        ``P``is replaced by :func:`convert_x_to_real_ft_coefficients` which only maps
        ``x`` to the positive frequencies of the Fourier Transform.
        Subsequently, the real inverse Fourier Transform is applied to the positive
        frequencies to obtain the time-domain signal.
        So basically, this is a shortcut to avoid the full mapping at all frequencies
        by exploiting the Hermite symmetry of the Fourier Transform for real-valued
        signals.

        """

        result = np.fft.irfft(
            convert_x_to_real_ft_coefficients(
                x=x,
                num_freqs=len(self.fourier_specs),
                x_real_imag_split_index=self.fourier_specs.x_real_imag_split_index,
                bandlimit_low_is_zero=self.fourier_specs.bandlimits.low_is_zero,
                freq_indices_in_nonzero_bandlimit_from=self.fourier_specs.freq_indices_in_nonzero_bandlimit_from,
                freq_indices_in_nonzero_bandlimit_to=self.fourier_specs.freq_indices_in_nonzero_bandlimit_to,
                freq_indices_zero_coeffs_below_low_to=self.fourier_specs.freq_indices_zero_coeffs_below_low_to,
            ),
            n=self.ft_size,
        )

        return self._apply_sqrt_weights(self._interpolate(result))

    def _rmatvec(self, x: np.ndarray) -> np.ndarray:
        """
        This is equivalent to the product ``P.T @ inv(F).T @ A.T @ sqrt(W).T @ y = P.T @ inv(F) @ A.T @ sqrt(W) @ y``
        where

        - ``sqrt(W)`` is the square root of the diagonal matrix of the weights
        - ``A.T`` is the transpose of the interpolation matrix that interpolates the
            signal values from the grid of time points to the time points where the
            signal is actually sampled
        - ``inv(F)`` is the Inverse Fourier Transform which is symmetric and thus equal
            to its transpose
        - ``P.T`` is the matrix that compresses the real and imaginary coefficients of
            the Fourier Transform to a compressed form as real values similar to the
            real ``x``-coefficients representing the Fourier Transform

        In contrast to :meth:`_matvec`, the full range of frequencies are required, so
        ``inv(F)`` cannot be replaced by the real Inverse Fourier Transform. Since ``y``
        is real, but not necessarily symmetric, the result of this operation will be
        complex and Hermitian symmetric.
        Now, ``P.T`` - which is replaced by :func:`compress_ft_coefficients_to_xlike` -
        takes the real and imaginary coefficients of the Fourier Transform at the
        frequencies within the bandlimit. These coefficients are then compressed by
        storing a concatenation of the two-fold of the real coefficients and the
        negative twofold of the imaginary coefficients in a compressed form as real
        values. If the zero frequency is included, the real coefficient for the zero
        frequency is stored as is at the beginning of the compressed vector.

        """  # noqa: E501

        result = np.fft.ifft(
            self._interpolate_transposed(self._apply_sqrt_weights(x)),
            n=self.ft_size,
        )

        return compress_ft_coefficients_to_xlike(
            x_real_ft=result,
            x_vect_size=self.fourier_specs.x_vect_size,
            x_real_imag_split_index=self.fourier_specs.x_real_imag_split_index,
            bandlimit_low_is_zero=self.fourier_specs.bandlimits.low_is_zero,
            freq_indices_in_nonzero_bandlimit_from=self.fourier_specs.freq_indices_in_nonzero_bandlimit_from,
            freq_indices_in_nonzero_bandlimit_to=self.fourier_specs.freq_indices_in_nonzero_bandlimit_to,
        )


class BandlimitedDecimationProblem:
    """
    A convenience class to solve bandlimited decimation problems.

    Parameters
    ----------
    t_values, y_values : :class:`numpy.ndarray` of shape (n,)
        The time points and the corresponding signal values.
        ``t_values`` don't have to be equidistantly spaced, but they have to be sorted.
    decimation_factor : :class:`float`
        The factor by which the number of time points is reduced.
        It has to be ``> 1.0``.
    bandlimit_high : :class:`float`
        The upper bandlimit frequency of the Fourier Transform.
    bandlimit_low : :class:`float`, default=0.0
        The lower bandlimit frequency of the Fourier Transform.
    y_uncertainties : :class:`numpy.ndarray` of shape (n,) or ``None``, default=``None``
        The uncertainties of the signal values at the time points which can be used for
        a Weighted Least Squares problem.
        If ``None``, the signal values are assumed to have equal uncertainties and an
        Ordinary Least Squares problem is solved.
    interpolate_polynomial_degree : {1, 3, 5}, default=3
        The degree of the polynomial used for interpolation.
    verbose : :class:`bool`, default=``False``
        If ``True``, additional information is printed during the computation when
        the method :meth:`fit` is called.

    """

    def __init__(
        self,
        t_values: np.ndarray,
        y_values: np.ndarray,
        decimation_factor: float,
        bandlimit_high: float,
        bandlimit_low: float = 0.0,
        y_uncertainties: Optional[np.ndarray] = None,
        interpolate_polynomial_degree: Literal[1, 3, 5] = 3,
        verbose: bool = False,
    ) -> None:

        # the required attributes are stored
        self.t_values: np.ndarray = t_values
        self.y_values: np.ndarray = y_values
        self.decimation_factor: float = decimation_factor
        self.bandlimit_high: float = bandlimit_high
        self.bandlimit_low: float = bandlimit_low
        self.y_uncertainties: Optional[np.ndarray] = y_uncertainties
        self.interpolate_polynomial_degree: Literal[1, 3, 5] = (
            interpolate_polynomial_degree
        )
        self.verbose: bool = verbose

        self.t_grid_for_decimation: np.ndarray
        self.linear_operator: LinearOperatorForBandlimitedDecimation
        self.right_hand_side_vector: np.ndarray
        self.solution: np.ndarray
        self.solution_covariance_main_diagonal: np.ndarray
        self.y_grid_decimated: np.ndarray
        self.is_set_up: bool = False
        self.is_fitted: bool = False

    def _setup_linear_system(self) -> None:
        """
        Sets up the linear system for bandlimited decimation.

        """

        # the decimation factor is validated
        if self.decimation_factor <= 1.0:
            raise ValueError(
                f"The decimation factor must be greater than 1.0, but got "
                f"{self.decimation_factor:.5e}."
            )

        # the grid of equally spaced time points for decimation is created
        self.t_grid_for_decimation: np.ndarray = get_t_grid_for_decimation(
            t_values=self.t_values,
            decimation_factor=self.decimation_factor,
            use_fft_next_fast_len=True,
        )

        # the linear operator for bandlimited decimation is created
        self.linear_operator: LinearOperatorForBandlimitedDecimation = (
            LinearOperatorForBandlimitedDecimation(
                t_values=self.t_values,
                t_grid_for_ft=self.t_grid_for_decimation,
                bandlimit_high=self.bandlimit_high,
                bandlimit_low=self.bandlimit_low,
                y_uncertainties=self.y_uncertainties,
                interpolate_polynomial_degree=self.interpolate_polynomial_degree,
            )
        )

        # finally, the right-hand side vector is computed
        self.right_hand_side_vector: np.ndarray = (
            self.linear_operator._apply_sqrt_weights(self.y_values)
        )

        self.is_set_up = True

        return

    def _find_initial_guess(self) -> np.ndarray:
        """
        Finds an initial guess for the solution of the bandlimited decimation problem.

        """

        # for the initial guess, the signal is interpolated to the grid values and its
        # Fourier Transform is compressed to an x-like vector
        interpolated_signal = np.interp(
            x=self.t_grid_for_decimation,
            xp=self.t_values,
            fp=self.y_values,
        )
        interpolated_signal_ft = np.fft.rfft(
            interpolated_signal,
            n=self.linear_operator.ft_size,
        )

        x0 = compress_ft_coefficients_to_xlike(
            x_real_ft=interpolated_signal_ft,
            x_vect_size=self.linear_operator.fourier_specs.x_vect_size,
            x_real_imag_split_index=self.linear_operator.fourier_specs.x_real_imag_split_index,
            bandlimit_low_is_zero=self.linear_operator.fourier_specs.bandlimits.low_is_zero,
            freq_indices_in_nonzero_bandlimit_from=self.linear_operator.fourier_specs.freq_indices_in_nonzero_bandlimit_from,
            freq_indices_in_nonzero_bandlimit_to=self.linear_operator.fourier_specs.freq_indices_in_nonzero_bandlimit_to,
        )

        # since the compression applies a factor of 2 to the real coefficients and a
        # factor of -2 to the imaginary coefficients, the initial guess has to be
        # scaled accordingly
        x_offset_index = 0 if not self.linear_operator.fourier_specs.bandlimits.low_is_zero else 1
        x0[x_offset_index : self.linear_operator.fourier_specs.x_real_imag_split_index] /= 2.0
        x0[self.linear_operator.fourier_specs.x_real_imag_split_index : x0.size] /= -2.0

        return x0

    def fit(self) -> "BandlimitedDecimationProblem":
        """
        Solves the bandlimited decimation problem.

        """

        # if the linear system is not set up yet, it is set up
        if not self.is_set_up:
            self._setup_linear_system()

        # an initial guess for the solution is found
        x0 = self._find_initial_guess()

        # the solution is found by solving the linear system
        solution = lsqr(
            A=self.linear_operator,
            b=self.right_hand_side_vector,
            x0=x0,
            calc_var=True,
            show=self.verbose,
        )

        self.solution = solution[0]
        self.solution_covariance_main_diagonal = solution[9]

        # the decimated signal is reconstructed by applying the linear operator to the
        # solution
        self.y_grid_decimated = np.fft.irfft(
            convert_x_to_real_ft_coefficients(
                x=self.solution,
                num_freqs=len(self.linear_operator.fourier_specs),
                x_real_imag_split_index=self.linear_operator.fourier_specs.x_real_imag_split_index,
                bandlimit_low_is_zero=self.linear_operator.fourier_specs.bandlimits.low_is_zero,
                freq_indices_in_nonzero_bandlimit_from=self.linear_operator.fourier_specs.freq_indices_in_nonzero_bandlimit_from,
                freq_indices_in_nonzero_bandlimit_to=self.linear_operator.fourier_specs.freq_indices_in_nonzero_bandlimit_to,
                freq_indices_zero_coeffs_below_low_to=self.linear_operator.fourier_specs.freq_indices_zero_coeffs_below_low_to,
            ),
            n=self.linear_operator.ft_size,
        )

        self.is_fitted = True

        return self

    def predict(self, t_values: np.ndarray) -> np.ndarray:
        """
        Predicts the signal values at the time points ``t_values``.

        Parameters
        ----------
        t_values : :class:`numpy.ndarray` of shape (n,)
            The time points at which the signal values should be predicted.

        Returns
        -------
        y_values : :class:`numpy.ndarray` of shape (n,)
            The predicted signal values.

        Raises
        ------
        ValueError
            If the method :meth:`fit` has not been called yet.

        """

        if not self.is_fitted:
            raise ValueError("The method `fit` has to be called before `predict`.")

        # the solution at the grid of time points for decimation is interpolated to the
        # time points where the signal values should be predicted
        interp_matrix = make_interpolation_matrix(
            t_grid=self.t_grid_for_decimation,
            t_eval=t_values,
            poly_degree=self.interpolate_polynomial_degree,
        )
        return interp_matrix @ self.y_grid_decimated


decimation_problem = BandlimitedDecimationProblem(
    t_values=t_values_regular,
    y_values=y_values_regular,
    decimation_factor=10.0,
    bandlimit_low=400.0,
    bandlimit_high=10_000.0,
    y_uncertainties=None,
    interpolate_polynomial_degree=3,
    verbose=True,
)
decimated_signal = decimation_problem.fit().predict(t_values=t_values_regular)




# convert_x_to_real_ft_coefficients_jit = jit(nopython=True)(
#     convert_x_to_real_ft_coefficients
# )
# compress_real_ft_coefficients_to_xlike_jit = jit(nopython=True)(
#     compress_ft_coefficients_to_xlike
# )
# second_order_differences_jit = jit(nopython=True)(second_order_differences)
# second_order_differences_transposed_jit = jit(nopython=True)(
#     second_order_differences_transposed
# )

# t = np.linspace(0.0, 1.0, 10)
# fourier_specs = get_fourier_specs(t, 0.0, 3.0)
# x = np.arange(1, fourier_specs.x_vect_size + 1).astype(np.float64)
# x_real_ift = convert_x_to_real_ft_coefficients(
#     x,
#     len(fourier_specs),
#     fourier_specs.x_real_imag_split_index,
#     fourier_specs.bandlimits.low_is_zero,
#     fourier_specs.freq_indices_in_nonzero_bandlimit_from,
#     fourier_specs.freq_indices_in_nonzero_bandlimit_to,
#     fourier_specs.freq_indices_zero_coeffs_below_low_to,
# )
# x_real_ift_jit = convert_x_to_real_ft_coefficients_jit(
#     x,
#     len(fourier_specs),
#     fourier_specs.x_real_imag_split_index,
#     fourier_specs.bandlimits.low_is_zero,
#     fourier_specs.freq_indices_in_nonzero_bandlimit_from,
#     fourier_specs.freq_indices_in_nonzero_bandlimit_to,
#     fourier_specs.freq_indices_zero_coeffs_below_low_to,
# )

# x_reconstructed = compress_ft_coefficients_to_xlike(
#     x_real_ift,
#     fourier_specs.x_vect_size,
#     fourier_specs.x_real_imag_split_index,
#     fourier_specs.bandlimits.low_is_zero,
#     fourier_specs.freq_indices_in_nonzero_bandlimit_from,
#     fourier_specs.freq_indices_in_nonzero_bandlimit_to,
# )
# x_reconstructed_jit = compress_real_ft_coefficients_to_xlike_jit(
#     x_real_ift,
#     fourier_specs.x_vect_size,
#     fourier_specs.x_real_imag_split_index,
#     fourier_specs.bandlimits.low_is_zero,
#     fourier_specs.freq_indices_in_nonzero_bandlimit_from,
#     fourier_specs.freq_indices_in_nonzero_bandlimit_to,
# )

In [ ]:
y = np.random.rand(t.size)
y_diff = second_order_differences(y)
assert np.allclose(y_diff, np.convolve(y, [1, -2, 1], mode="valid"))
y_diff = second_order_differences_transposed(y)
assert y.size == y_diff.size
assert np.allclose(y_diff, np.convolve(np.pad(y, (1, 1)), [1, -2, 1], mode="valid"))
%timeit convert_x_to_real_ift_coefficients(x, len(fourier_specs), fourier_specs.x_real_imag_split_index, fourier_specs.bandlimits.low_is_zero, fourier_specs.freq_indices_in_nonzero_bandlimit_from, fourier_specs.freq_indices_in_nonzero_bandlimit_to, fourier_specs.freq_indices_zero_coeffs_below_low_to)
%timeit convert_x_to_real_ift_coefficients_jit(x, len(fourier_specs.freqs), fourier_specs.x_real_imag_split_index, fourier_specs.bandlimits.low_is_zero, fourier_specs.freq_indices_in_nonzero_bandlimit_from, fourier_specs.freq_indices_in_nonzero_bandlimit_to, fourier_specs.freq_indices_zero_coeffs_below_low_to)
%timeit compress_real_ift_coefficients_to_xlike(x_real_ift, fourier_specs.x_vect_size, fourier_specs.x_real_imag_split_index, fourier_specs.bandlimits.low_is_zero, fourier_specs.freq_indices_in_nonzero_bandlimit_from, fourier_specs.freq_indices_in_nonzero_bandlimit_to)
%timeit compress_real_ift_coefficients_to_xlike_jit(x_real_ift, fourier_specs.x_vect_size, fourier_specs.x_real_imag_split_index, fourier_specs.bandlimits.low_is_zero, fourier_specs.freq_indices_in_nonzero_bandlimit_from, fourier_specs.freq_indices_in_nonzero_bandlimit_to)
print("Timing derivatives")
%timeit np.convolve(np.pad(y, (2, 2)), [1, -2, 1], mode="valid")
%timeit second_order_differences(y)
%timeit second_order_differences_jit(y)
%timeit second_order_differences_transposed(y)
%timeit second_order_differences_transposed_jit(y)

In [ ]:
plt.close("all")

fig, ax = plt.subplots(nrows=2, sharex=True)
fig2, ax2 = plt.subplots()

ax[0].plot(t_values_regular, y_values_regular, "o", label="Original Signal")
ax[0].plot(t_values_regular, decimated_signal, label="Decimated Signal")

ax[1].fill_between(
    t_values_regular,
    -2.0 * noise_stddevs_irregular,
    2.0 * noise_stddevs_irregular,
    color="gray",
    alpha=0.5,
    label="Noise",
)
ax[1].plot(t_values_regular, y_values_regular - decimated_signal, label="Residuals")

ax2.plot(
    np.fft.rfftfreq(n=t_values_regular.size, d=t_values_regular[1] - t_values_regular[0]),
    np.abs(np.fft.rfft(y_values_regular, norm="forward")),
)
ax2.plot(
    np.fft.rfftfreq(n=decimation_problem.t_grid_for_decimation.size, d=decimation_problem.t_grid_for_decimation[1] - decimation_problem.t_grid_for_decimation[0]),
    np.abs(np.fft.rfft(decimation_problem.y_grid_decimated, norm="forward")),
)



In [ ]:
t_values_fast, y_values_fast = prepare_signal_for_fast_fft(
    t_grid=t_values_regular,
    y_values=y_values_regular + 5e-4 * np.random.randn(y_values_regular.size),
)

linop = BandlimitedIFT(
    t_grid=t_values_fast,
    bandlimit_low=0.0,
    bandlimit_high=6_000.0,
    # bandlimit_high=101243.0,
)

x0 = 0.5 * compress_real_ft_coefficients_to_xlike_jit(
    x_real_ft=np.fft.fft(y_values_fast),
    x_vect_size=linop.fourier_specs.x_vect_size,
    x_real_imag_split_index=linop.fourier_specs.x_real_imag_split_index,
    bandlimit_low_is_zero=linop.fourier_specs.bandlimits.low_is_zero,
    freq_indices_in_nonzero_bandlimit_from=linop.fourier_specs.freq_indices_in_nonzero_bandlimit_from,
    freq_indices_in_nonzero_bandlimit_to=linop.fourier_specs.freq_indices_in_nonzero_bandlimit_to,
)
x0[linop.fourier_specs.x_real_imag_split_index:] *= -1.0

print(np.max(np.abs(np.fft.irfft(convert_x_to_real_ft_coefficients_jit(
    x=x0,
    num_freqs=len(linop.fourier_specs),
    x_real_imag_split_index=linop.fourier_specs.x_real_imag_split_index,
    bandlimit_low_is_zero=linop.fourier_specs.bandlimits.low_is_zero,
    freq_indices_in_nonzero_bandlimit_from=linop.fourier_specs.freq_indices_in_nonzero_bandlimit_from,
    freq_indices_in_nonzero_bandlimit_to=linop.fourier_specs.freq_indices_in_nonzero_bandlimit_to,
    freq_indices_zero_coeffs_below_low_to=linop.fourier_specs.freq_indices_zero_coeffs_below_low_to,
), n=y_values_fast.size) - y_values_fast)))

bandlimited_fft = lsqr(
    A=linop,
    b=y_values_fast,
    x0=x0,
    show=True,
)

%timeit lsqr(A=linop, b=y_values_fast, x0=x0)

In [ ]:
plt.close("all")

fig, ax = plt.subplots()
fig2, ax2 = plt.subplots(nrows=2, sharex=True)

solution = convert_x_to_real_ft_coefficients(
    bandlimited_fft[0],
    len(linop.fourier_specs),
    linop.fourier_specs.x_real_imag_split_index,
    linop.fourier_specs.bandlimits.low_is_zero,
    linop.fourier_specs.freq_indices_in_nonzero_bandlimit_from,
    linop.fourier_specs.freq_indices_in_nonzero_bandlimit_to,
    linop.fourier_specs.freq_indices_zero_coeffs_below_low_to,
)

signal_reconstructed = np.fft.irfft(
    solution,
    n=t_values_fast.size,
)
# signal_reconstructed = linop @ x0

ax.plot(
    np.fft.rfftfreq(n=t_values_fast.size, d=t_values_fast[1] - t_values_fast[0]),
    np.abs(np.fft.rfft(y_values_fast)),
    lw=4,
)
ax.plot(
    np.fft.rfftfreq(n=t_values_fast.size, d=t_values_fast[1] - t_values_fast[0]),
    np.abs(solution),
)

ax2[0].plot(t_values_fast, y_values_fast, label="Original Signal")
ax2[0].plot(t_values_fast, signal_reconstructed, label="Reconstructed Signal")

ax2[1].plot(t_values_fast, y_values_fast - signal_reconstructed)

In [ ]:
a = np.linspace(0, 1, 101)
b = np.sin(4 * np.pi * a) + np.sin(8 * np.pi * a)
nfn = next_fast_len(a.size, real=True)

a_freqs = np.fft.rfftfreq(n=a.size, d=a[1] - a[0])
a_freqs_nfn = np.fft.rfftfreq(n=nfn, d=a[1] - a[0])

b_fft = np.fft.rfft(b, n=a.size)
b_fft_nfn = np.fft.rfft(b, n=nfn)

fig, ax = plt.subplots()

ax.plot(a_freqs, np.abs(b_fft))
ax.plot(a_freqs_nfn, np.abs(b_fft_nfn))

In [ ]:
def second_order_differences(y: np.ndarray) -> np.ndarray:
    """
    Computes the second-order differences of the given vector y.

    Parameters
    ----------
    y : :class:`numpy.ndarray` of shape (m,)
        The data for which the second-order differences are computed.

    Returns
    -------
    y_diff : :class:`numpy.ndarray` of shape (m-2,)
        The second-order differences of ``y``.

    """

    y_diff = np.empty(shape=(y.size - 2,), dtype=np.float64)
    for i in range(y_diff.size):
        y_diff[i] = y[i + 2] - 2.0 * y[i + 1] + y[i]

    return y_diff


def second_order_differences_transposed(y: np.ndarray) -> np.ndarray:
    """
    Computes the second order differences of the given array y when the finite
    difference matrix ``D`` in the product is transposed.
    This is equivalent to padding ``y`` with 1 leading and trailing zeros and applying
    the second order differences with flipped coefficients (has no effect in this case).

    Parameters
    ----------
    y : :class:`numpy.ndarray` of shape (m,)
        The data for which the second-order differences are computed.

    Returns
    -------
    y_diff : :class:`numpy.ndarray` of shape (m,)
        The second-order differences of ``y``.

    """

    y_diff = np.empty(shape=(y.size,), dtype=np.float64)
    y_diff[0] = -2.0 * y[0] + y[1]
    for i in range(1, y_diff.size - 1):
        y_diff[i] = y[i - 1] - 2.0 * y[i] + y[i + 1]

    y_diff[y_diff.size - 1] = y[y.size - 2] - 2.0 * y[y.size - 1]

    return y_diff